# P3 Fuse — ICNet-inspired CFF

One notebook: select control or ex1–ex5 in the first config cell.
Four CFF stages, standard dilated deep Conv+BN, skip per experiment, Add/ReLU,
fixed P2 mids, P2 bilinear x2+pad/align_corners=True. Single head, no CLG/DS.
Clone f/fuse-cff → attach PhenoBench → GPU T4×2 → Save & Run All → one ZIP.
P2 recipe:100epochs/val5/FP32/seed1682409321/batch4perGPU/global8.
Mandatory native train/val and DDP resource gates; stop on OOM, no fallback.
Exports are unfused with exact strict reload checks; reparameterization is optional.
Validation sampling retains P2 drop_last (768/772); full evaluation is separate.


In [ ]:
# FIRST CONFIG CELL — select one of five approved experiments.
REPO_URL = 'https://github.com/FUAVX/RPD_Reproduce.git'
GIT_REF = 'f/fuse-cff'
SOURCE_COMMIT = ''  # optional reachable commit
EXPERIMENT_MODE = 'ex1'  # ex1=FullD2, ex2=Dila1, ex3=Dila3, ex4=BNonly, ex5=IdenSkip
RUN_TAG = 'run1'
DATASET_DIR = ''
NUM_WORKERS = 2
DEVICES = 2
WORKING_ROOT = '/kaggle/working'
RUN_BENCHMARKS = True  # longer disposable measurement; mandatory resource gates remain enabled
EXPORT_MAIN = True
EXPERIMENTS = {'control': 'CFF-Control', 'ex1': 'CFF-Ex1', 'ex2': 'CFF-Ex2', 'ex3': 'CFF-Ex3', 'ex4': 'CFF-Ex4', 'ex5': 'CFF-Ex5'}
if EXPERIMENT_MODE not in EXPERIMENTS:
    raise ValueError('Select control or ex1–ex5')
if DEVICES != 2:
    raise ValueError('P2 runtime requires T4 x2')
SCENARIO = EXPERIMENTS[EXPERIMENT_MODE]


In [ ]:
# Kaggle supplies torch/CUDA; retain that installation.
!nvidia-smi
!pip install -q 'numpy<2' 'pytorch-lightning==1.9.5' 'torchmetrics==0.11.4' 'setuptools<70' tensorboard matplotlib pandas tabulate timm scikit-image tifffile albumentations pyyaml rich


In [ ]:
import json
import pathlib
import shutil
import subprocess
import yaml
WORKING_DIR = pathlib.Path(WORKING_ROOT)
REPO_DIR = WORKING_DIR / 'RPD_P2'
RUN_DIR = WORKING_DIR / f'{SCENARIO}_{RUN_TAG}'
if RUN_DIR.exists():
    raise FileExistsError(f'Output already exists: {RUN_DIR}; choose a new RUN_TAG')
def run(command, cwd=None):
    print('+', ' '.join(map(str, command)))
    subprocess.run(command, cwd=cwd, check=True)
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
run(['git', 'clone', '--branch', GIT_REF, '--single-branch', REPO_URL, str(REPO_DIR)])
if SOURCE_COMMIT:
    run(['git', 'checkout', '--detach', SOURCE_COMMIT], cwd=REPO_DIR)
BASE_COMMIT = 'd77d518b8f650443216070fa76742db76e41b8ac'
run(['git', 'merge-base', '--is-ancestor', BASE_COMMIT, 'HEAD'], cwd=REPO_DIR)
scenario_path = REPO_DIR / 'configs/p3_fuse_cff' / f'{SCENARIO}.yaml'
required_paths = [scenario_path] + [REPO_DIR / name for name in (
    'test_wdice.py', 'test_cff.py', 'test_validation_infrastructure.py',
    'models/rpdnet/fuse_cff.py', 'utils/cff_campaign.py', 'utils/validation_artifacts.py',
    'scripts/check_cff_run.py', 'scripts/benchmark_cff.py', 'scripts/profile_cff.py',
    'scripts/export_cff.py', 'scripts/check_cff_data.py', 'scripts/cff_smoke.py', 'scripts/preflight_cff_resources.py',
    'callbacks/cff_cost.py', 'utils/tensorboard_compat.py', 'tests/reference/cff/sources.json')]
missing = [str(path) for path in required_paths if not path.is_file()]
if missing:
    raise FileNotFoundError('Prepared CFF source missing: ' + ', '.join(missing))
def is_phenobench_root(path):
    return (
        (path / 'train' / 'images').is_dir()
        and (path / 'train' / 'semantics').is_dir()
        and (path / 'val' / 'images').is_dir()
        and (path / 'val' / 'semantics').is_dir()
    )

if DATASET_DIR:
    dataset_dir = pathlib.Path(DATASET_DIR)
else:
    kaggle_input = pathlib.Path('/kaggle/input')
    candidates = [
        kaggle_input / 'datasets' / 'ndhoang2310' / 'phenobench-dataset' / 'PhenoBench',
        kaggle_input / 'phenobench-dataset' / 'PhenoBench',
        kaggle_input / 'datasets' / 'ndhoang2310' / 'phenobench-dataset',
        kaggle_input / 'phenobench-dataset',
    ]
    if kaggle_input.is_dir():
        attached_inputs = [p for p in kaggle_input.iterdir() if p.is_dir()]
        candidates.extend(p / 'PhenoBench' for p in attached_inputs)
        candidates.extend(attached_inputs)
    dataset_dir = next((p for p in candidates if is_phenobench_root(p)), None)
if dataset_dir is None or not is_phenobench_root(dataset_dir):
    raise FileNotFoundError('Cannot find a complete PhenoBench root in /kaggle/input. Attach the PhenoBench dataset, or set DATASET_DIR to its root containing train/images, train/semantics, val/images, and val/semantics.')

print(f'[OK] source ref: {subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip()}')
print(f'[OK] dataset: {dataset_dir}')
print(f'[OK] train images: {len(list((dataset_dir / "train" / "images").glob("*")))}')


In [ ]:
# Runtime overrides follow the successful WDice notebook.
import copy
import hashlib
RUN_DIR.mkdir(parents=True, exist_ok=True)
cfg = yaml.safe_load(scenario_path.read_text())
base_path = REPO_DIR / 'configs/p2_scenarios/P2_Ex7_RepDWNet_WDice-Loss.yaml'
base_cfg = yaml.safe_load(base_path.read_text())
inherited = copy.deepcopy(cfg)
cff = inherited['backbone'].pop('cff', None)
inherited['experiment'] = base_cfg['experiment']
assert inherited == base_cfg, 'Recipe drift beyond metadata/CFF'
expected_variants = {'ex1': (2, 'projection_bn'), 'ex2': (1, 'projection_bn'), 'ex3': (3, 'projection_bn'), 'ex4': (2, 'bn_only'), 'ex5': (2, 'identity')}
if EXPERIMENT_MODE == 'control':
    assert cff is None
else:
    expected_dilation, expected_skip = expected_variants[EXPERIMENT_MODE]
    assert cff == dict({'variant': 'cff', 'decoder_mid_channels': [128, 64, 32, 16], 'align_corners': True, 'resize': 'bilinear_x2_pad', 'conv_bias': False, 'conv_init': 'kaiming_normal_fan_in_a0', 'bn_eps': 1e-05, 'bn_momentum': 0.1, 'bn_affine': True, 'bn_track_running_stats': True, 'bn_gamma': 1.0, 'bn_beta': 0.0}, dilation=expected_dilation, skip_mode=expected_skip)
assert cfg['train']['max_epoch'] == 100
assert cfg['val']['check_val_every_n_epoch'] == 5
assert cfg['train']['precision'] == 32 and cfg['train']['batch_size'] == 4
assert cfg['train']['loss'] == 'weighted_dice'
cfg['data']['path_to_dataset'] = str(dataset_dir)
cfg['data']['num_workers'] = NUM_WORKERS
active_cfg_path = RUN_DIR / f'active_config_{SCENARIO}.yaml'
active_cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False))
runtime_cfg = copy.deepcopy(cfg)
runtime_cfg['train']['devices'] = DEVICES
runtime_cfg['val']['devices'] = 1  # exact legacy --devices override
runtime_cfg_path = RUN_DIR / 'runtime_config.yaml'
runtime_cfg_path.write_text(yaml.safe_dump(runtime_cfg, sort_keys=False))
print(yaml.safe_dump(runtime_cfg, sort_keys=False))


In [ ]:
PREFLIGHT_PASSED = False
# Preflight only; disposable processes never mutate the real campaign.
import torch
if not torch.cuda.is_available() or torch.cuda.device_count() != DEVICES:
    raise RuntimeError('Choose Kaggle GPU T4 x2')
gpu_names = [torch.cuda.get_device_name(i) for i in range(DEVICES)]
if any('T4' not in name for name in gpu_names):
    raise RuntimeError(f'Expected T4 x2, got {gpu_names}')
run(['python', 'scripts/check_cff_data.py', '--dataset-dir', str(dataset_dir),
     '--output', str(RUN_DIR / 'dataset_audit.json')], cwd=REPO_DIR)
for script in ['test_wdice.py', 'test_validation_infrastructure.py', 'test_cff.py']:
    run(['python', script], cwd=REPO_DIR)
infrastructure_smoke = ['python', 'test_validation_infrastructure.py', '--ddp',
    '--output-dir', str(RUN_DIR / 'smoke/validation_infrastructure'),
    '--accelerator', 'gpu', '--devices', str(DEVICES)]
run(infrastructure_smoke, cwd=REPO_DIR)
run(infrastructure_smoke + ['--resume'], cwd=REPO_DIR)
selected_modes = list(dict.fromkeys(['control', EXPERIMENT_MODE]))
for mode in selected_modes:
    smoke = ['python', 'test_cff.py', '--smoke', '--config',
             str(REPO_DIR / 'configs/p3_fuse_cff' / f'{EXPERIMENTS[mode]}.yaml'),
             '--output-dir', str(RUN_DIR / 'smoke' / mode), '--accelerator', 'gpu',
             '--devices', str(DEVICES)]
    run(smoke, cwd=REPO_DIR)
    run(smoke + ['--resume'], cwd=REPO_DIR)
run(['python', 'test_cff.py', '--pipeline-smoke', '--config', str(scenario_path),
     '--output-dir', str(RUN_DIR / 'smoke/pipeline'), '--accelerator', 'gpu',
     '--devices', str(DEVICES)], cwd=REPO_DIR)
shutil.make_archive(str(RUN_DIR / 'smoke/pipeline_evidence'), 'zip', root_dir=RUN_DIR / 'smoke/pipeline')
shutil.rmtree(RUN_DIR / 'smoke/pipeline')
run(['python', 'scripts/profile_cff.py', '--device', 'cuda', '--experiments', *selected_modes,
     '--output', str(RUN_DIR / 'complexity_native_shapes.json')], cwd=REPO_DIR)
run(['python', 'scripts/preflight_cff_resources.py', '--device', 'cuda', '--dataset-dir', str(dataset_dir),
     '--experiments', *selected_modes, '--output', str(RUN_DIR / 'native_resource_audit.json')], cwd=REPO_DIR)
# Mandatory native DDP resource smoke, even if long benchmarks are disabled.
run(['python', '-m', 'torch.distributed.run', '--standalone', '--nproc_per_node=2',
     'scripts/benchmark_cff.py', '--mode', 'training', '--device', 'cuda',
     '--experiments', *selected_modes, '--warmup', '1', '--iterations', '2',
     '--output-dir', str(RUN_DIR / 'smoke/native_ddp')], cwd=REPO_DIR)
if RUN_BENCHMARKS:
    run(['python', '-m', 'torch.distributed.run', '--standalone', '--nproc_per_node=2',
         'scripts/benchmark_cff.py', '--mode', 'training', '--device', 'cuda',
         '--experiments', *selected_modes, '--output-dir', str(RUN_DIR / 'benchmark')], cwd=REPO_DIR)
    run(['python', 'scripts/benchmark_cff.py', '--mode', 'inference', '--device', 'cuda',
         '--experiments', *selected_modes, '--output-dir', str(RUN_DIR / 'benchmark')], cwd=REPO_DIR)

PREFLIGHT_PASSED = True


In [ ]:
if not globals().get('PREFLIGHT_PASSED', False):
    raise RuntimeError('Preflight must pass before the real campaign; no recipe fallback is permitted')
resource_audit = json.loads((RUN_DIR / 'native_resource_audit.json').read_text())
if resource_audit['status'] != 'passed':
    raise RuntimeError('Native resource audit failed; stop and report to team')
# Real training starts from scratch in a new process with the original P2 recipe.
TRAIN_LOG_DIR = RUN_DIR / 'training_output'
train_command = ['python', 'multi_metric_train.py', '--config', str(active_cfg_path),
                 '--export_dir', str(TRAIN_LOG_DIR), '--devices', str(DEVICES)]
run(train_command, cwd=REPO_DIR)


In [ ]:
# Require synchronized checkpoint/history/selection parity before completing ZIP.
run(['python', 'scripts/check_cff_run.py', '--output-dir', str(TRAIN_LOG_DIR),
     '--report', str(RUN_DIR / 'campaign_audit.json')], cwd=REPO_DIR)
if EXPORT_MAIN:
    for policy in ['best_mIoU', 'best_weedIoU']:
        matches = sorted((TRAIN_LOG_DIR / 'checkpoints').glob(f'*{policy}*.ckpt'))
        if len(matches) != 1:
            raise RuntimeError(f'Expected exactly one {policy} checkpoint: {matches}')
        run(['python', 'scripts/export_cff.py', '--config', str(runtime_cfg_path),
             '--checkpoint', str(matches[0]), '--output-dir', str(RUN_DIR / 'main_export' / policy),
             '--device', 'cuda'], cwd=REPO_DIR)
import importlib.metadata
source_commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip()
manifest = {
    'experiment': SCENARIO, 'mode': EXPERIMENT_MODE, 'run_tag': RUN_TAG,
    'source_repository': REPO_URL, 'requested_ref': GIT_REF, 'requested_commit': SOURCE_COMMIT,
    'source_commit': source_commit, 'base_commit': BASE_COMMIT,
    'cff': cff, 'base_config': str(base_path.relative_to(REPO_DIR)),
    'base_config_sha256': hashlib.sha256(base_path.read_bytes()).hexdigest(),
    'resolved_config': runtime_cfg, 'train_command': train_command,
    'batch_size_per_GPU': cfg['train']['batch_size'], 'global_batch_size': cfg['train']['batch_size'] * DEVICES,
    'gpu_names': gpu_names, 'bn': 'standard BatchNorm2d, sync_batchnorm=False',
    'seed': cfg['seed'], 'same_production_criterion': True, 'single_head_validation': True,
    'initialization': {'common_host':'P2 encoder/head retained; CFF and replacement decoder inside RNG fork/restore',
                       'CFF_conv':'Kaiming normal fan_in/a0; biasFalse',
                       'CFF_BN':'eps1e-5/momentum0.1/affineTrue/track_running_statsTrue/gamma1/beta0',
                       'RepDW':'native P2 constructor initialization', 'seed':cfg['seed']},
    'regularization':'P2 coupled Adam WD2e-4, no new parameter groups or extra regularizer',
    'official_reference': {'repo':'https://github.com/hszhao/ICNet',
        'commit':'dd2bf78ff3f9ff05f6d35c40a3ee216158d9e892',
        'file':'evaluation/prototxt/icnet_cityscapes.prototxt',
        'scope':'evaluation graph; training BN/bias/init are team policies'},
    'resize_policy':'team adaptation: P2 bilinear x2 + pad, align_corners=True; differs from Caffe 2H-1 zoom',
    'resource_audit': resource_audit, 'automatic_recipe_fallback': False,
    'benchmarks_requested': RUN_BENCHMARKS, 'main_export_requested': EXPORT_MAIN,
    'versions': {name: importlib.metadata.version(name) for name in
                 ('torch', 'torchvision', 'pytorch-lightning', 'torchmetrics', 'numpy')},
    'validation_infrastructure': {'D01': 'preflight stops on255; metric semantics unchanged',
        'D02': 'global synchronized metrics; rank-zero persistent writer',
        'D03': 'checkpoint-owned history; no sanity rows; deduplication'},
    'validation_sampling': {'drop_last': True, 'batch_size_per_rank': cfg['val']['batch_size'],
        'world_size': DEVICES, 'note': 'P2 sampling retained:768/772; independent full evaluation is separate'},
    'dataset_audit': json.loads((RUN_DIR / 'dataset_audit.json').read_text()),
    'campaign_audit': json.loads((RUN_DIR / 'campaign_audit.json').read_text()),
    'status': 'completed',
}
required_outputs = [TRAIN_LOG_DIR / 'checkpoints/last.ckpt', TRAIN_LOG_DIR / 'val_history.csv',
                    TRAIN_LOG_DIR / 'training_cost.json', runtime_cfg_path, RUN_DIR / 'dataset_audit.json',
                    RUN_DIR / 'campaign_audit.json', RUN_DIR / 'complexity_native_shapes.json', RUN_DIR / 'native_resource_audit.json', RUN_DIR / 'smoke/native_ddp/benchmark_training.json']
if RUN_BENCHMARKS:
    required_outputs.extend(RUN_DIR / 'benchmark' / f'benchmark_{mode}.json' for mode in ['training', 'inference'])
if EXPORT_MAIN:
    required_outputs.extend(RUN_DIR / 'main_export' / policy / 'main_state_dict.pt'
                            for policy in ['best_mIoU', 'best_weedIoU'])
missing = [str(path) for path in required_outputs if not path.is_file()]
if missing:
    raise FileNotFoundError('Missing completed artifacts: ' + ', '.join(missing))
(RUN_DIR / 'run_manifest.json').write_text(json.dumps(manifest, indent=2))
archive = shutil.make_archive(str(WORKING_DIR / f'{SCENARIO}_{RUN_TAG}_artifacts'), 'zip', root_dir=RUN_DIR)
print(f'[DONE] Download: {archive}')
print(f'[DONE] Source: {source_commit}')
